# Embeddings from three RNA language models (RNA-FM, ERNIE-RNA, RiNALMo)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Upload `mature.fa` and `hairpin.fa` (miRBase). Run the cells top to bottom. Each model's files are saved as `<model>_<mature|hairpin>_embeddings.npy` + `_names.csv`; at the end everything is zipped and downloaded.

RiNALMo has 650M parameters and is the slowest; the notebook runs it in float16.

## 1. Install

In [ ]:
!pip install -q multimolecule biopython

## 2. Upload mature.fa and hairpin.fa

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

## 3. Setup

In [ ]:
import gc, numpy as np, pandas as pd, torch
import multimolecule
from multimolecule import RnaTokenizer, RnaFmModel, ErnieRnaModel, RiNALMoModel
from Bio import SeqIO

SPECIES_PREFIX, MAX_LEN = "hsa-", 1022
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODELS = {
    "rnafm":    ("multimolecule/rnafm",    RnaFmModel,    32),
    "ernierna": ("multimolecule/ernierna", ErnieRnaModel, 32),
    "rinalmo":  ("multimolecule/rinalmo-giga",  RiNALMoModel,  8),
}

def load_fasta(path):
    recs = [(r.id, str(r.seq).upper().replace("T", "U")[:MAX_LEN])
            for r in SeqIO.parse(path, "fasta") if r.id.startswith(SPECIES_PREFIX)]
    print(f"{path}: {len(recs)} human sequences")
    return recs

FASTA = {"mature": load_fasta("mature.fa"), "hairpin": load_fasta("hairpin.fa")}

## 4. Embedding function (mean-pool over nucleotide tokens)

In [ ]:
@torch.no_grad()
def embed(model, tokenizer, records, bs):
    out = []
    for i in range(0, len(records), bs):
        seqs = [s for _, s in records[i:i + bs]]
        enc = tokenizer(seqs, padding=True, return_tensors="pt", return_special_tokens_mask=True)
        special = enc.pop("special_tokens_mask").bool()
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            h = model(**enc).last_hidden_state.float()
        mask = (enc["attention_mask"].bool() & ~special.to(device)).unsqueeze(-1).float()
        out.append(((h * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

## 5. Run all models

In [ ]:
for tag, (hf_id, cls, bs) in MODELS.items():
    print(f"\n=== {tag} ({hf_id}) ===")
    try:
        tok = RnaTokenizer.from_pretrained(hf_id)
        model = cls.from_pretrained(hf_id).to(device).eval()
        for kind, recs in FASTA.items():
            E = embed(model, tok, recs, bs)
            assert not np.isnan(E).any(), "NaN in embeddings"
            np.save(f"{tag}_{kind}_embeddings.npy", E)
            pd.DataFrame({"name": [n for n, _ in recs], "length": [len(s) for _, s in recs]}).to_csv(
                f"{tag}_{kind}_names.csv", index=False)
            print(f"  saved {tag}_{kind}_embeddings.npy  shape={E.shape}")
    except Exception as e:
        print(f"  FAILED: {e}")
    finally:
        model = None; gc.collect(); torch.cuda.empty_cache()

## 6. Download

In [ ]:
!zip -q rna_lm_embeddings.zip *_embeddings.npy *_names.csv
files.download("rna_lm_embeddings.zip")